# Bài tập: Ứng dụng Decision Tree trong Feature Engineering 
Bài tập này sẽ giới thiệu đến các bạn một ứng dụng khác của Decision Tree đó là Feature Engineering. 

## Mục tiêu
- Hiểu các kỹ thuật Feature Engineering
- Sử dụng Decision Trees để chọn ra những đặc trưng thực sự quan trọng.

## Tìm hiểu về Feature Engineering 

Feature engineering là một bước quan trọng trong quá trình xây dựng mô hình machine learning, giúp cải thiện hiệu suất của mô hình bằng cách tạo ra các đặc trưng (features) tốt hơn từ dữ liệu gốc. Decision Tree là một công cụ mạnh mẽ có thể được sử dụng trong feature engineering để phát hiện và tạo ra các đặc trưng mới. 

Một số ứng dụng của feature engineering có thể kể đến như: phát hiện các mối quan hệ phi tuyến, tạo các đặc trưng tương tác, giảm số lượng đặc trưng,...

Trong bài tập này, chúng ta sẽ tìm hiểu về kỹ thuật giảm số lượng đặc trưng bằng Decision Tree, cụ thể: 
- Decision Tree có thể giúp chọn ra các đặc trưng quan trọng nhất, giảm thiểu số lượng đặc trưng cần thiết trong mô hình.
- Ứng dụng: Sử dụng Decision Tree để xác định và giữ lại các đặc trưng quan trọng, loại bỏ những đặc trưng ít hoặc không quan trọng, giúp giảm độ phức tạp của mô hình và tăng hiệu suất.

Ý tưởng của việc tính toán mức độ quan trọng của một đặc trưng là dựa trên mức độ ảnh hưởng của đặc trưng đó lên kết quả phân loại. Hình dung tại mỗi node trong cây quyết định, chúng ta có một giá trị đo lường độ "hỗn loạn" (impurity) của node đó. Khi mô hình sử dụng một đặc trưng để chia một node thành hai nhóm con, nó tính toán độ hỗn loạn trước và sau khi chia. Nếu độ hỗn loạn giảm nhiều sau khi chia thì đặc trưng được sử dụng cho việc chia này được coi là quan trọng. 

Mô hình cây quyết định thực hiện phân chia nhiều lần trên toàn bộ cây, sử dụng nhiều đặc trưng khác nhau. Để tính toán tầm quan trọng của mỗi đặc trưng, mô hình cộng tất cả các sự giảm thiểu "độ hỗn loạn" do đặc trưng đó tạo ra trên toàn bộ cây. Cuối cùng, các tầm quan trọng này được chuẩn hóa để tổng bằng 1, tạo ra giá trị `feature_importances_`.

Có nhiều metric khác nhau dùng để đánh giá "độ hỗn loạn" tại một node, có thể kể đến như Gini Impurity hoặc Entropy. Ở bài này, chúng ta sẽ tìm hiểu về Gini Impurity. Dưới đây là thuật toán minh họa cho quá trình tính độ quan trọng của các đặc trưng. 

1. **Tính Gini Impurity cho Node hiện tại:**
   - **Gini Impurity** được tính theo công thức:
     $$
     G(p) = 1 - \sum_{i=1}^{C} p_i^2
     $$
     Trong đó:
     - $ p_i $ là tỉ lệ của các mẫu thuộc lớp $ i $ trong node.
     - $ C $ là số lượng lớp.
   - Minh họa bằng code:
      ```python
      import numpy as np
      from collections import Counter

      # Hàm tính Gini Impurity cho một tập hợp dữ liệu
      def gini_impurity(y):
         # Đếm số lượng mỗi lớp trong node
         class_counts = Counter(y)
         total_samples = len(y)
         
         # Tính tỉ lệ cho mỗi lớp
         impurity = 1.0
         for cls in class_counts:
            prob_cls = class_counts[cls] / total_samples
            impurity -= prob_cls ** 2
         
         return impurity
      ```

2. **Tính Gini Impurity cho các node con:**
   - Với mỗi ngưỡng chia (threshold), chia dữ liệu tại node hiện tại thành hai node con (left node và right node).
   - Tính Gini Impurity cho từng Node con:

     $$
     G_{\text{split}} = \frac{N_{\text{left}}}{N_{\text{total}}} G_{\text{left}} + \frac{N_{\text{right}}}{N_{\text{total}}} G_{\text{right}}
     $$

     Trong đó:
     - $ N_{\text{left}} $ và $ N_{\text{right}} $ là số lượng mẫu trong Node con bên trái và bên phải.
     - $ G_{\text{left}} $ và $ G_{\text{right}} $ là Gini Impurity của Node con bên trái và bên phải.
     - $ N_{\text{total}} $ là tổng số lượng mẫu trong Node hiện tại.

3. **Tính feature importance ở node hiện tại:**
   - Độ quan trọng của feature $j$ tại node $t$ có thể được đánh giá bởi mức độ giảm "độ hỗn loạn" bởi công thức 
   $$
   I_{j, t} = G_t - \frac{N_{left}}{N_{t}}G_{left} - \frac{N_{right}}{N_t}G_{right}
   $$
   - Độ quan trọng của feature $j$ trên toàn bộ cây sẽ bằng tổng độ quan trọng ở tất cả các node: 
   $$
   I_{j} = \sum_{t \in \text{nodes where feature $j$ is used}}I_{j, t}
   $$
   - Minh họa bằng code:
      ```python
      # Hàm tính feature importance cho một node
      def feature_importance_node(y, left_y, right_y):
         n = len(y)
         n_left = len(left_y)
         n_right = len(right_y)

         gini_current = gini_impurity(y)
         gini_left = gini_impurity(left_y)
         gini_right = gini_impurity(right_y)

         # Tính mức độ giảm độ hỗn loạn
         importance = gini_current - (n_left/n) * gini_left - (n_right/n) * gini_right
         return importance

      # Hàm chính để minh họa tính toán trên
      def calculate_feature_importances(X, y, thresholds):
         n_samples, n_features = X.shape
         feature_importances = np.zeros(n_features)

         for feature_index in range(n_features):
            for threshold in thresholds[feature_index]:
               left_indices = X[:, feature_index] <= threshold
               right_indices = X[:, feature_index] > threshold
               
               if sum(left_indices) > 0 and sum(right_indices) > 0:
                  left_y = y[left_indices]
                  right_y = y[right_indices]
                  
                  importance = feature_importance_node(y, left_y, right_y)
                  feature_importances[feature_index] += importance

         # Chuẩn hóa feature importance
         feature_importances /= np.sum(feature_importances)
         return feature_importances
      ```

Trên đây là minh họa thủ công, trong thực tế, chúng ta có thể gọi đến thuộc tính `feature_importances_` của class DecisionTreeClassifier để lấy ra được độ quan trọng của các đặc trưng mà không cần phải cài đặt lại như vậy. Phần bài tập phía dưới các bạn hãy tìm hiểu và sử dụng thuộc tính này nhé!


## Bộ dữ liệu CICMalDroid 2020

Bộ dữ liệu CICMalDroid 2020 được cung cấp bởi Canada-India Research Center for AI (CICAI), là một nguồn dữ liệu về mã độc Android, đã được thu thập từ nhiều nguồn khác nhau từ tháng 12 năm 2017 đến tháng 12 năm 2018. Tổng số mẫu thu thập là hơn 17,341 ứng dụng Android ở dạng file apk. Dữ liệu trong bộ dữ liệu này đã được phân tích động bằng hệ thống phân tích động dựa trên ảo hóa máy chủ CopperDroid. Từ tổng số 17,341 mẫu, khoảng 13,077 mẫu đã chạy thành công và được phân tích. 

Qua quá trình tiền xử lý dữ liệu thu được file csv với 11598 mẫu dữ liệu, mỗi mẫu chứa 471 đặc trưng liên quan đến hành vi của các ứng dụng Android. Các đặc trưng này là các hành vi hoặc quyền truy cập của ứng dụng, được biểu diễn dưới dạng số nguyên.

Bây giờ ta sẽ đi vào đọc file `feature_vector.csv` và quan sát một số đặc điểm của dữ liệu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('feature_vector.csv')
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.info

In [ ]:
df.describe()

In [ ]:
label_counts = df['Class'].value_counts()

labels = label_counts.index.tolist()
counts = label_counts.tolist()

plt.bar(labels, counts)
plt.xlabel('Malware')
plt.ylabel('Counts')
plt.title('Malware Distribution in Dataset')
plt.show()

### Thực hiện phân loại bằng Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

# Tách dữ liệu thành các tập huấn luyện và kiểm tra
X = df.drop('Class', axis=1)
y = df['Class'] - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Khởi tạo mô hình Decision Tree
# 1 line of code 
# dt_model = ... (random_state=42)
# *** START CODE HERE ***

# *** END CODE HERE ***

# Huấn luyện mô hình
dt_model.fit(X_train, y_train)

# Dự đoán trên tập kiểm tra
y_pred = dt_model.predict(X_test)

# Đánh giá mô hình
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(accuracy)
print(class_report)

In [ ]:
# TEST YOUR CODE HERE
assert str(round(accuracy, 4)) == '0.9043'

In [ ]:
import seaborn as sns

sns.heatmap(conf_matrix , cmap='YlGnBu', fmt='', xticklabels=['Adware' ,'Banking' ,'SMS malware', 'Riskware','Benign'], yticklabels=['Adware' ,'Banking' ,'SMS malware', 'Riskware','Benign'], annot=True)

### Thực hiện feature engineering (giảm số lượng đặc trưng) bằng Decision Tree

Để thực hiện việc giảm số lượng đặc trưng bằng Decision Tree, ta có thể thực hiện theo các bước sau:

- Bước 1: Huấn luyện mô hình Decision Tree.

    Huấn luyện một mô hình Decision Tree (`dt_model`) trên tập dữ liệu.

- Bước 2: Lấy tầm quan trọng của các đặc trưng.

    Mô hình Decision Tree có thuộc tính `feature_importances_`, đây là một mảng chứa giá trị tầm quan trọng của từng đặc trưng. Tầm quan trọng của các đặc trưng được tính toán dựa trên mức độ giảm độ lệch chuẩn (impurity) mà mỗi đặc trưng mang lại khi nó được sử dụng để phân chia dữ liệu trong cây quyết định. Giá trị này càng cao thì đặc trưng đó càng quan trọng.

- Bước 3: Sắp xếp và chọn các đặc trưng quan trọng.

    Dựa trên các giá trị tầm quan trọng, bạn có thể sắp xếp các đặc trưng và chọn những đặc trưng có tầm quan trọng cao hơn một ngưỡng nào đó.

- Bước 4: Tạo lại tập dữ liệu chỉ bao gồm các đặc trưng quan trọng.

    Sử dụng các đặc trưng đã chọn để tạo lại tập dữ liệu giảm số lượng đặc trưng.

- Bước 5: Huấn luyện lại mô hình trên tập dữ liệu đã giảm số lượng đặc trưng và đánh giá hiệu suất.

    Huấn luyện lại mô hình Decision Tree và đánh giá hiệu suất trên tập dữ liệu giảm số lượng đặc trưng.

In [ ]:
# Lấy tầm quan trọng của các đặc trưng từ dt_model
# *** START CODE HERE ***
feature_importances = ...
# *** END CODE HERE ***

# Lấy danh sách các đặc trưng
# *** START CODE HERE ***
features = ...
# *** END CODE HERE ***

# Tạo DataFrame tầm quan trọng của các đặc trưng
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})

# Sắp xếp các đặc trưng theo tầm quan trọng giảm dần
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# TEST YOUR CODE HERE
assert round(feature_importances[0], 4) == 0.0370
assert round(feature_importances[1], 4) == 0

In [ ]:
# Vẽ biểu đồ tầm quan trọng của các đặc trưng
plt.figure(figsize=(12, 8))
plt.barh(feature_importance_df['Feature'][:20], feature_importance_df['Importance'][:20])
plt.gca().invert_yaxis()
plt.title('Top 20 Important Features')
plt.xlabel('Importance')
plt.show()

In [ ]:
# Chọn ngưỡng tầm quan trọng để loại bỏ các đặc trưng không cần thiết 
threshold = 0.01
important_features = feature_importance_df[feature_importance_df['Importance'] > threshold]['Feature']
print(f"Number of important features: {important_features.shape[0]}")

# Tạo lại tập dữ liệu chỉ bao gồm các đặc trưng quan trọng
# *** START CODE HERE ***
X_train_reduced = ...
X_test_reduced = ...
# *** END CODE HERE ***

# Huấn luyện lại mô hình trên tập dữ liệu đã giảm số lượng đặc trưng
# random_state=42
# *** START CODE HERE ***
dt_model_reduced = ...
dt_model_reduced.fit(...)
# *** END CODE HERE ***

# Dự đoán trên tập kiểm tra
y_pred_reduced = dt_model_reduced.predict(X_test_reduced)

# Đánh giá mô hình
accuracy_reduced = accuracy_score(y_test, y_pred_reduced)
conf_matrix_reduced = confusion_matrix(y_test, y_pred_reduced)
class_report_reduced = classification_report(y_test, y_pred_reduced)

print("Accuracy:", accuracy_reduced)
print("Classification Report:\n", class_report_reduced)

In [ ]:
assert str(round(accuracy_reduced, 4)) == '0.9003'

In [ ]:
sns.heatmap(conf_matrix_reduced , cmap='YlGnBu', fmt='', xticklabels=['Adware' ,'Banking' ,'SMS malware', 'Riskware','Benign'], yticklabels=['Adware' ,'Banking' ,'SMS malware', 'Riskware','Benign'], annot=True)

## Nhận xét

Chỉ với số đặc trưng giảm xuống gần 30 lần nhưng hiệu suất mô hình giảm xuống không đáng kể, điều đó cho thấy sự mạnh mẽ của Decision Tree trong Feature Engineering. Các thông số cụ thể như sau: 

- Accuracy: Độ chính xác tổng thể của mô hình Decision Tree sau khi giảm số lượng đặc trưng từ 470 xuống còn 16 là 90.03%, chỉ giảm nhẹ so với mô hình ban đầu (90.43%).
- Precision và Recall: Precision và recall cho các lớp vẫn cao, cho thấy việc giảm số lượng đặc trưng không ảnh hưởng nhiều đến hiệu suất của mô hình.
- F1-score: F1-score cho tất cả các lớp đều cao và gần như không thay đổi so với mô hình ban đầu.


## Kết luận

Việc giảm số lượng đặc trưng không làm giảm đáng kể hiệu suất của mô hình, đồng thời giúp mô hình trở nên nhẹ hơn và dễ hiểu hơn. Điều này cho thấy các đặc trưng quan trọng đã được giữ lại và các đặc trưng ít quan trọng hơn đã được loại bỏ. 

# Bài tập: Manual Decision Tree Splitting

## Mục tiêu

- Cài đặt from scratch các hàm `entropy()`, `information_gain()` để xác định cách chia tốt nhất tại một node trong Decision Tree.
## Tìm hiểu về Entropy và Information Gain

Trong lĩnh vực học máy và khoa học dữ liệu, **Entropy** và **Information Gain** là hai khái niệm quan trọng giúp chúng ta đo lường sự không chắc chắn và lượng thông tin thu được từ dữ liệu. Để hiểu rõ hơn về hai khái niệm này, chúng ta sẽ đi từ những khái niệm cơ bản nhất: **probability**, **surprise**, và sau đó mở rộng đến **entropy** và **information gain**.

### 1. Từ Probability đến Surprise  

#### Probability (Xác suất)
Xác suất là một khái niệm cơ bản trong thống kê, biểu diễn khả năng xảy ra của một sự kiện. Ví dụ, nếu bạn tung một đồng xu đồng chất đồng lượng, xác suất để nhận được mặt ngửa (head) là 0.5, và xác suất để nhận được mặt sấp (tail) cũng là 0.5.

#### Surprise (Sự bất ngờ)
Khi một sự kiện xảy ra, mức độ "bất ngờ" của nó phụ thuộc vào xác suất của sự kiện đó. Nếu một sự kiện có xác suất cao (ví dụ: mặt trời mọc vào buổi sáng), chúng ta sẽ không cảm thấy bất ngờ khi nó xảy ra. Ngược lại, nếu một sự kiện có xác suất thấp (ví dụ: trúng xổ số), chúng ta sẽ cảm thấy rất bất ngờ.

Có thể mường tượng, surprise là một đại lượng tỉ lệ nghịch với xác suất của sự kiện. Công thức tính supprise khi đó có thể hình dung là $\text{Surprise}(x) = \frac{1}{P(x)}$. Tuy nhiên, công thức này có điểm hạn chế là khi xác suất của sự kiện bằng 1, tức sự kiện đó chắc chắn xảy ra, thì surprise đáng lý ra nên bằng 0 để thể hiện việc không có gì bất ngờ, trong khi với công thức hiện tại thì surprise sẽ bằng 1, một điều không hợp lý cho lắm. Để giải quyết vấn đề này, người ta thêm vào hàm logarit một số cơ số $n$ nào đó, thường là 2, để đưa miền giá trị của surprise về $(0, +\infty)$. Khi đó ta có công thức: $\text{Surprise}(x) = \log_2(\frac{1}{P(x)}) = -\log_2(P(x))$.


### 2. Từ Surprise đến Entropy

#### Entropy 
Entropy là một khái niệm trong lý thuyết thông tin, được sử dụng để đo lường **sự không chắc chắn** hoặc **sự hỗn loạn** của một hệ thống. Trong bối cảnh của một biến ngẫu nhiên, entropy đo lường lượng thông tin trung bình mà bạn nhận được khi biết kết quả của biến đó. Bạn có thể hình dung entropy chính là trung bình có trọng số của surprise cho mỗi giá trị mà biến ngẫu nhiên đó có thể nhận, trọng số ở đây bằng đúng xác suất xảy ra sự kiện tương ứng. Công thức tính entropy cho một biến ngẫu nhiên $ X $ nhận giá trị thuộc tập $\left \{  x_1, x_2, \dots, x_n \right \}$ khi đó là:
$$
H(X) = -\sum_{i=1}^{n} P(x_i) \cdot \log_2(P(x_i))
$$
Trong đó:
- $ P(x_i) $ là xác suất của giá trị $ x_i $.
- $ H(X) $ là entropy của biến $ X $.

*Ví dụ*: Nếu bạn có một đồng xu công bằng, entropy của nó là:
$$
H(X) = - (0.5 \cdot \log_2(0.5) + 0.5 \cdot \log_2(0.5)) = 1 
$$

### 3. Từ Entropy Đến Information Gain

#### Information Gain 
Trong học máy, **Information Gain** là một khái niệm quan trọng được sử dụng để xác định tính hiệu quả của một đặc trưng trong việc phân chia dữ liệu. Information Gain đo lường sự giảm entropy sau khi chia dữ liệu dựa trên một đặc trưng cụ thể.

Công thức tính information gain khi chia dữ liệu dựa trên đặc trưng $ A $ là:
$$
\text{IG}(A) = H(Y) - H(Y|A)
$$
Trong đó:
- $ H(Y) $ là entropy của biến mục tiêu $ Y $ (ví dụ: sống sót hay không).
- $ H(Y|A) $ là entropy có điều kiện của $ Y $ sau khi chia dữ liệu dựa trên đặc trưng $ A $.

*Ví dụ*: Nếu bạn chia dữ liệu Titanic dựa trên giới tính (nam/nữ), information gain sẽ cho biết mức độ hiệu quả của việc sử dụng giới tính để dự đoán khả năng sống sót.


### Cài đặt hàm tính entropy

Viết hàm tính entropy cho một mảng 1 chiều `y`, biết rằng giá trị entropy được tính theo công thức: 

$$\text{entropy}(y) = -\sum_{i=1}^{n} p_i \log_2(p_i)$$

Trong đó:
- $y$ là một mảng 1 chiều chứa giá trị của biến mục tiêu.
- $n$ là số lượng phần tử phân biệt trong mảng `y`.
- $p_i$ là tỉ lệ giữa số lần xuất hiện của giá trị $i$ trong mảng `y` và tổng số phần tử trong mảng `y`.

**Ví dụ 1:** 

Input | Output |
---------|----------|
y = [1, 2, 2, 3, 3, 3]| 1.459148 |


Giải thích: 

- Mảng ```y``` có 3 giá trị phân biệt là 1, 2, 3.
- Xác suất của từng giá trị là: 
    - $p(1) = 1/6$
    - $p(2) = 2/6$
    - $p(3) = 3/6$
- Tính entroypy của ```y```:    
 $-\left(\frac{1}{6} \log_2 \frac{1}{6} + \frac{2}{6} \log_2 \frac{2}{6} + \frac{3}{6} \log_2 \frac{3}{6} \right) = 1.459148$

 **Ví dụ 2:**

Input | Output |
---------|----------|
y = [5, 5, 5, 5]| 0 |

Giải thích: Vì các giá trị trong mảng ```y``` đều bằng nhau nên entropy trong trường hợp này bằng 0.

In [ ]:
import numpy as np  

def entropy(y: np.ndarray[tuple[int]]) -> float:  
    '''
    Calculate the entropy of a label array.
    Args: 
        y: a numpy array of shape (n_samples, )

    Returns:
        The entropy of y
    '''
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###
# Test  
y = np.array([0, 1, 0, 1, 1])  
assert np.isclose(entropy(y), 0.97095), "Something is wrong when calculate entropy! Please try again!"
print(entropy(y))  # Output: ~0.9709  

## Viết hàm chia dữ liệu `X` dựa trên đặc trưng `feature`

Viết hàm `split_data()` nhận đầu vào là một numpy array `X` chứa dữ liệu có kích thước (n_samples, n_features) với `n_samples` là số lượng mẫu và `n_features` là số lượng đặc trưng, một `feature_idx` thể hiện chỉ số của cột trong `X` mà chúng ta muốn chia dữ liệu. Hàm trả về một dictionary chứa dữ liệu đã phân chia, với key là các giá trị phân biệt của cột `feature_idx` và value là mảng chứa các chỉ số tương ứng với giá trị đó.

Ví dụ: 

| **Input** | **Output** |
|-----------|-----------|
| X = np.array([ <br> [1, 2], <br> [3, 4], <br> [1, 3], <br> [3, 2]]) <br> feature_idx = 0 | {1: [0, 2], 3: [1, 3]} |


Giải thích: 

Với `feature_idx = 0`, dễ thấy có hai giá trị phân biệt là 1 và 3 trong cột 0 của `X`. Ta đi xác định các chỉ số dòng tương ứng chứa giá trị 1 và 3 ở cột 0, cụ thể:

    - Dòng 0 và 2 của `X` có giá trị 1 ở cột 0.
    - Dòng 1 và 3 của `X` có giá trị 3 ở cột 0.

In [ ]:
def split_data(X: np.ndarray[tuple[int, int]], feature_idx: int) -> dict[int, list[int]]:  
    '''
    Split the data X based on a feature.
    Args: 
        X: a numpy array of shape (n_samples, n_features)  
        feature_idx: the feature index to split on
    
    Returns:
        A dictionary containing the split data
        The dictionary key is the unique value of the feature
        The value is a list of indices of the data rows
    '''
    
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###

# Test  
X = np.array([  
    [0, 0],  
    [0, 1],  
    [1, 0],  
    [1, 1],  
    [1, 0],  
])  
splits = split_data(X, 0)  
assert splits == {0: [0, 1], 1: [2, 3, 4]}, "Something is wrong when split data! Please try again!"
print(splits)  

## Cài đặt hàm tính Information Gain 

Hãy cài đặt hàm `information_gain()` nhận đầu vào là một numpy array `X` chứa dữ liệu, một numpy array `y` chứa giá trị của biến mục tiêu và một `feature_idx` thể hiện chỉ số của đặc trưng mà ta muốn tính Information Gain. Hàm trả về giá trị Information Gain khi chia dữ liệu dựa trên đặc trưng `feature_idx`.

Như đã trình bày ở trên, Information Gain được tính bằng công thức:

$$
\text{IG} = H(Y) - H(Y|X) = H(Y) - \sum_{i=1}^{n} \frac{N_i}{N} H(Y_i)
$$

Trong đó:
- $i$ là chỉ số của các nhóm
- $N_i$ là số lượng mẫu thuộc nhóm $i$
- $N$ là tổng số mẫu
- $H(Y_i)$ là entropy của nhóm $i$.


Ví dụ: Giả sử chúng ta có một bộ dữ liệu đơn giản về việc có nên đi chơi hay không dựa trên điều kiện thời tiết:

| Weather (0: Sunny, 1: Rainy) | Temperature (0: Cool, 1: Hot) | Go Out (0: No, 1: Yes) |
|------------------------------|------------------------------|----------------------|
| 0 (Sunny)                    | 0 (Cool)                     | 0 (No)               |
| 0 (Sunny)                    | 1 (Hot)                      | 1 (Yes)              |
| 1 (Rainy)                    | 0 (Cool)                     | 0 (No)               |
| 1 (Rainy)                    | 1 (Hot)                      | 1 (Yes)              |
| 1 (Rainy)                    | 0 (Cool)                     | 1 (Yes)              |

Hãy tính Information Gain cho cả hai đặc trưng Weather và Temperature để xem đặc trưng nào tốt hơn cho việc quyết định có nên đi chơi hay không:

1. **Weather (Sunny/Rainy)**:
    - Khi Weather = Sunny (2 mẫu): [0, 1] → Entropy = 1
    - Khi Weather = Rainy (3 mẫu): [0, 1, 1] → Entropy ≈ 0.918
    - IG ≈ 0.02

2. **Temperature (Cool/Hot)**:
    - Khi Temperature = Cool (3 mẫu): [0, 0, 1] → Entropy ≈ 0.918
    - Khi Temperature = Hot (2 mẫu): [1, 1] → Entropy = 0
    - IG ≈ 0.42

Như vậy, Temperature có Information Gain cao hơn, cho thấy đây là đặc trưng tốt hơn để quyết định có nên đi chơi hay không.

In [ ]:
def information_gain(X: np.ndarray[tuple[int, int]], y: np.ndarray[tuple[int, int]], feature_idx: int) -> float:  
    ''' 
    Calculate the information gain.
    Args: 
        X: a numpy array of shape (n_samples, n_features)  
        y: a numpy array of shape (n_samples, )  
        feature_idx: the feature index to calculate information gain
    
    Returns:
        The information gain of the feature
    '''
    ### START CODE HERE ###
    
    
    ### END CODE HERE ###

# Test  
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
    [1, 0],
])
y = np.array([0, 1, 0, 1, 1])  
ig_weather = information_gain(X, y, 0)  
ig_temp = information_gain(X, y, 1)  
print(f"IG for Weather: {ig_weather}")  # ~0.02
print(f"IG for Temperature: {ig_temp:.2f}")  # ~0.42  
assert np.isclose(ig_weather, 0.02, atol=.01), "Something is wrong when calculate information gain! Please try again!"
assert np.isclose(ig_temp, 0.42, atol=.01), "Something is wrong when calculate information gain! Please try again!"

## Chọn ra đặc trưng tốt nhất để phân chia

Dựa vào những hàm đã cài đặt bên trên, chúng ta sẽ chọn ra đặc trưng tốt nhất để phân chia dữ liệu tại một node trong Decision Tree. Đặc trưng tốt nhất sẽ là đặc trưng có Information Gain lớn nhất.

In [ ]:
def best_split(X: np.ndarray[tuple[int, int]], y: np.ndarray[int]):  
    '''
    Find the best feature to split.
    Args: 
        X: a numpy array of shape (n_samples, n_features)  
        y: a numpy array of shape (n_samples, )
    Returns:
        The index of the best feature to split on
    '''
    ### START CODE HERE ###
    
     
    ### END CODE HERE ###

# Test  
best_feature = best_split(X, y)  
assert best_feature == 1, "Something is wrong when find the best feature to split! Please try again!"
print(f"Best feature to split on: {best_feature}")  

## Hoàn thành 

Bạn đã hoàn thành xong các yêu cầu của và cũng vừa tìm hiểu xong về Entropy và Information Gain. Hy vọng rằng bài tập này sẽ giúp bạn hiểu rõ hơn về cách Decision Tree hoạt động. 

Trong thực tế, để xây dựng một Decision Tree, chúng ta sẽ sử dụng các thư viện như `scikit-learn` để giúp việc cài đặt nhanh chóng và hiệu quả hơn. Khi đó, chúng ta không cần phải cài đặt các hàm này từ đầu mà có thể sử dụng các hàm/tham số có sẵn trong thư viện. 

Ví dụ như đoạn code bên dưới sẽ tạo ra một Decision Tree với các tham số mặc định, sử dụng entropy làm tiêu chí phân chia dữ liệu.
```python
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(criterion='entropy')
clf.fit(X_train, y_train)
```

Chúc mừng bạn đã hoàn thành bài tập. Hãy thử áp dụng kiến thức đã học vào các bài toán thực tế khác nhau.